# RSNA Knee Arm CNXT — gold58 OOF predictions

Runs goodpjw2008's ConvNeXt 2.5D reader (3 released folds; gold never trained,
their protocol) on the 58 gold-labelled train studies for the stacking harness.
Writes `arm_cnxt_58.csv` (study × 12 label probabilities) + per-label gold AUC.


In [ ]:
import glob, os, sys, time
import numpy as np, pandas as pd, pydicom
print('python', sys.version.split()[0], '| pydicom', pydicom.__version__)
for mod in ('pylibjpeg', 'gdcm', 'PIL'):
    try:
        __import__(mod); print(f'{mod}: available')
    except Exception:
        print(f'{mod}: missing')
CMP = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
D = None
for _c in ('/kaggle/input/datasets/goodpjw2008/rsna-knee-2-5d-convnext-reader',
           '/kaggle/input/goodpjw2008-rsna-knee-2-5d-convnext-reader'):
    if os.path.isdir(_c):
        D = _c; break
assert D, 'cnxt dataset not mounted'
print('cnxt dir:', D)
print('cnxt files:', sorted(os.listdir(D)))
st0 = sorted(os.listdir(f'{CMP}/train_series'))[0]
sr0 = sorted(os.listdir(f'{CMP}/train_series/{st0}'))[0]
f0 = sorted(os.listdir(f'{CMP}/train_series/{st0}/{sr0}'))[0]
probe = f'{CMP}/train_series/{st0}/{sr0}/{f0}'
print('probe file:', f0)
try:
    ds = pydicom.dcmread(probe)
    arr = ds.pixel_array
    ts = str(getattr(getattr(ds, 'file_meta', None), 'TransferSyntaxUID', 'no-meta'))
    print('DECODE OK', arr.shape, arr.dtype, '| ts:', ts, '| photo:', ds.get('PhotometricInterpretation', '?'))
except Exception as e:
    print('DECODE FAIL:', repr(e))

In [ ]:
sys.path.insert(0, D)
from infer import run
from knee import LABELS
from sklearn.metrics import roc_auc_score

tr = pd.read_csv(f'{CMP}/train.csv')
gold = tr[tr['ACL'].notna()].copy()
studies = gold.StudyInstanceUID.tolist()
print('gold studies:', len(studies))
ckpts = sorted(glob.glob(f'{D}/cnxt_v0_fold*.pt'))
print('ckpts:', [os.path.basename(p) for p in ckpts])
t0 = time.time()
pred = run(CMP, 'train', ckpts, studies=studies)
print(f'infer: {time.time()-t0:.0f}s | pred shape {pred.shape}')
pred.to_csv('/kaggle/working/arm_cnxt_58.csv', index=True)

y = gold.set_index('StudyInstanceUID').reindex(pred.index)
auc = {c: float(roc_auc_score(y[c].values, pred[c].values)) for c in LABELS}
print('gold macro AUC:', round(float(np.mean(list(auc.values()))), 4))
print({k: round(v, 3) for k, v in auc.items()})
print('DONE')